# Transformer 100M CUDA — Google Colab Training

**Sebelum mulai**, pastikan struktur Google Drive seperti ini:
```
MyDrive/
  transformer_100m_cuda.zip   ← source code
  data/
    corpus_final/
      english/...
      indonesian/...
  sft_data/
    sft_clean/...
    dpo/...
```

> **Urutan eksekusi:** Jalankan cell satu per satu dari atas ke bawah.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Config — Sesuaikan path di sini

In [ ]:
import os

# ╔══════════════════════════════════════════════════════════╗
# ║  SESUAIKAN PATH INI DENGAN LOKASI FILE DI GOOGLE DRIVE  ║
# ╚══════════════════════════════════════════════════════════╝
DRIVE_ROOT = "/content/drive/MyDrive"   # ganti jika ada subfolder, misal: "/content/drive/MyDrive/Projects"

ZIP_PATH   = f"{DRIVE_ROOT}/transformer_100m_cuda.zip"
DATA_PATH  = f"{DRIVE_ROOT}/data"        # folder berisi corpus_final/
SFT_PATH   = f"{DRIVE_ROOT}/sft_data"   # folder berisi sft_clean/, dpo/

# Working directory di Colab (lokal — lebih cepat dari Drive)
WORK_DIR   = "/content/transformer_100m_cuda"

# Checkpoint disimpan ke Drive agar persist saat Colab disconnect
CKPT_DRIVE = f"{DRIVE_ROOT}/checkpoints_100m"

# ── Export sebagai env vars agar bash cell bisa akses langsung ──
os.environ["WORK_DIR"]   = WORK_DIR
os.environ["ZIP_PATH"]   = ZIP_PATH
os.environ["DATA_PATH"]  = DATA_PATH
os.environ["SFT_PATH"]   = SFT_PATH
os.environ["CKPT_DRIVE"] = CKPT_DRIVE

# ── Validasi file ada di Drive ──
errors = []
if not os.path.exists(ZIP_PATH):
    errors.append(f"  ✗ ZIP tidak ditemukan : {ZIP_PATH}")
if not os.path.isdir(DATA_PATH):
    errors.append(f"  ✗ data/ tidak ditemukan: {DATA_PATH}")
if not os.path.isdir(SFT_PATH):
    errors.append(f"  ✗ sft_data/ tidak ditemukan: {SFT_PATH}")

if errors:
    print("❌ PATH SALAH — periksa kembali DRIVE_ROOT di atas:")
    for e in errors: print(e)
else:
    print("✓ Semua path ditemukan")
    print(f"  ZIP      : {ZIP_PATH}")
    print(f"  data/    : {DATA_PATH}")
    print(f"  sft_data/: {SFT_PATH}")
    print(f"  ckpt/    : {CKPT_DRIVE}")


## 3. Extract source code ke local disk (lebih cepat)

In [ ]:
import os, shutil

# Clean previous run jika ada
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
    print(f"Cleaned old {WORK_DIR}")

# Buat folder tujuan dulu, lalu unzip isinya langsung ke sana
os.makedirs(WORK_DIR, exist_ok=True)
!unzip -q "$ZIP_PATH" -d "$WORK_DIR"

# Verify
assert os.path.exists(f"{WORK_DIR}/CMakeLists.txt"), \
    f"Extraction failed! File yang ada di {WORK_DIR}:\n" + str(os.listdir(WORK_DIR)[:10])
print(f"✓ Source extracted to {WORK_DIR}")


## 4. Symlink data folders (avoid copying large files)

In [ ]:
import os, shutil

def make_symlink(target, link):
    if os.path.islink(link):
        os.unlink(link)
    elif os.path.isdir(link):
        shutil.rmtree(link)
    elif os.path.exists(link):
        os.remove(link)
    os.symlink(target, link)
    print(f"✓ {os.path.basename(link)}/ -> {target}")

# Symlink data dari Drive ke working dir (tidak perlu copy)
make_symlink(DATA_PATH,  f"{WORK_DIR}/data")
make_symlink(SFT_PATH,   f"{WORK_DIR}/sft_data")

# Checkpoint dir di Drive (persist across sessions)
os.makedirs(CKPT_DRIVE, exist_ok=True)
make_symlink(CKPT_DRIVE, f"{WORK_DIR}/checkpoints")


## 5. Check GPU & CUDA version

In [ ]:
!nvidia-smi
!nvcc --version

## 6. Build the project

In [ ]:
%%bash
cd "$WORK_DIR"

# Detect GPU type for optimal compilation
GPU_NAME=$(nvidia-smi --query-gpu=name --format=csv,noheader | head -1)
echo "Detected GPU: $GPU_NAME"

if echo "$GPU_NAME" | grep -qi "T4"; then
    GPU_TARGET="T4"
elif echo "$GPU_NAME" | grep -qi "V100"; then
    GPU_TARGET="V100"
elif echo "$GPU_NAME" | grep -qi "A100"; then
    GPU_TARGET="A100"
elif echo "$GPU_NAME" | grep -qi "L4"; then
    GPU_TARGET="RTX4090"   # L4 = sm_89, same as Ada Lovelace
elif echo "$GPU_NAME" | grep -qi "H100"; then
    GPU_TARGET="H100"
else
    GPU_TARGET="ALL"        # fallback: fat binary
fi

echo "Building for: $GPU_TARGET"
mkdir -p build && cd build
cmake .. -DCMAKE_BUILD_TYPE=Release -DGPU_TARGET=$GPU_TARGET 2>&1 | tail -5
make -j$(nproc) 2>&1 | tail -10
echo ""
echo "✓ Build complete"
ls -lh train_pretrain_cuda train_sft_cuda 2>/dev/null


## 7. Start Pre-training

In [ ]:
import subprocess, os

# ═══ TRAINING PARAMETERS ═══
# Sesuaikan batch_size berdasarkan GPU:
#   T4 (16GB)   : 64-75
#   L4 (24GB)   : 100-110
#   A100 (40GB) : 200
#   A100 (80GB) : 400

env = os.environ.copy()
env.pop('CUDA_VISIBLE_DEVICES', None)

os.makedirs(f'{WORK_DIR}/checkpoints', exist_ok=True)

print("▶ Starting pretraining...")
print("=" * 60)

proc = subprocess.Popen(
    ['stdbuf', '-oL', './build/train_pretrain_cuda',
     '--data_dir',       'data/corpus_final',
     '--vocab_dir',      'vocab_6',
     '--bpe',            'bpe_v6.bin',
     '--checkpoint_dir', 'checkpoints',
     '--batch_size',     '75',
     '--max_seq',        '256',
     '--max_tokens',     '2000000000',
     '--save_every',     '50',
     '--log_every',      '5',
     '--loss_scale',     '1024',
     '--debug_steps',    '30',
     '--resume'],
    cwd=WORK_DIR,
    env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

try:
    for line in proc.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    proc.terminate()
    print("\n⚠ Training dihentikan manual (Ctrl+C).")

proc.wait()
if proc.returncode == 0:
    print("\n✓ Training selesai.")
else:
    print(f"\n✗ Training exit code: {proc.returncode}")


## 7b. (Optional) Resume training jika Colab disconnect

Karena checkpoints disimpan di Drive, tinggal jalankan ulang dari Cell 1.
Flag `--resume` akan otomatis load checkpoint terakhir.

## 8. (Optional) Multi-GPU — DDP dengan 2 GPU

Colab Pro+ kadang bisa dapat 2 GPU. Jika ya, rebuild dengan DDP:

In [ ]:
import subprocess, sys, os

NUM_GPUS = int(subprocess.check_output("nvidia-smi -L | wc -l", shell=True).strip())
print(f"Available GPUs: {NUM_GPUS}")

if NUM_GPUS >= 2:
    print("Multi-GPU detected! Rebuilding with DDP...")

    # Install NCCL jika belum ada
    os.system("apt-get install -y libnccl2 libnccl-dev 2>/dev/null")

    # Rebuild dengan DDP
    build_cmd = (
        f"cd {WORK_DIR}/build && "
        f"cmake .. -DCMAKE_BUILD_TYPE=Release -DGPU_TARGET=T4 -DUSE_DDP=ON && "
        f"make -j$(nproc)"
    )
    ret = os.system(build_cmd)
    if ret != 0:
        raise RuntimeError("DDP build failed!")
    print("✓ DDP build complete")

    # Launch DDP training dengan streaming output
    cmd = [
        "bash", f"{WORK_DIR}/scripts/launch_ddp.sh",
        f"{WORK_DIR}/build/train_pretrain_cuda",
        "--data_dir",       "data/corpus_final",
        "--vocab_dir",      "vocab_6",
        "--checkpoint_dir", "checkpoints",
        "--batch_size",     "64",
        "--seq_len",        "256",
        "--max_tokens",     "2000000000",
        "--lr",             "3e-4",
        "--resume",
    ]

    print("=" * 60)
    proc = subprocess.Popen(
        cmd,
        cwd=WORK_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        universal_newlines=True,
        env={**os.environ, "NUM_GPUS": str(NUM_GPUS)},
    )
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
    except KeyboardInterrupt:
        proc.terminate()
        print("\n⚠ Training dihentikan manual.")
    proc.wait()

else:
    print("Single GPU terdeteksi — jalankan cell 7 (tanpa DDP)")


## 9. Monitor training progress

In [ ]:
# Check latest checkpoint
import glob
ckpts = sorted(glob.glob(f"{WORK_DIR}/checkpoints/*.safetensors"))
if ckpts:
    print(f"Latest checkpoint: {ckpts[-1]}")
    print(f"Total checkpoints: {len(ckpts)}")
else:
    print("No checkpoints yet")

# GPU utilization
!nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv